In [ ]:
"""
Label the support tickets with a DeepSeek teacher and build the chat-format SFT data.

The teacher's JSON labels are the gold standard the student is later trained and scored against.
"""
import pandas as pd
df = pd.read_csv("hf://datasets/Tobi-Bueck/customer-support-tickets/dataset-tickets-multi-lang-4-20k.csv")
print(df.shape)
print(df.columns.tolist())
print(df['priority'].value_counts(dropna=False))
print(df['queue'].value_counts())

# The source uses finer-grained queues than our five categories, so fold the
# extras in. IT Support and outages all become Technical Support, and so on.
queue_to_category = {
    "Technical Support": "Technical Support",
    "IT Support": "Technical Support",
    "Service Outages and Maintenance": "Technical Support",
    "Product Support": "Product Support",
    "Customer Service": "Customer Service",
    "General Inquiry": "Customer Service",
    "Human Resources": "Customer Service",
    "Billing and Payments": "Billing & Payments",
    "Sales and Pre-Sales": "Billing & Payments",
    "Returns and Exchanges": "Returns & Exchanges",
}
df["category"] = df["queue"].map(queue_to_category)
print(df["category"].value_counts())

from sklearn.model_selection import train_test_split

cols = ["subject", "body", "category", "priority", "language"]
data = df[cols].dropna(subset=["body", "category", "priority"]).reset_index(drop=True)

train_pool, test = train_test_split(
    data, test_size=1200, stratify=data["category"], random_state=42)

# Teacher labeling is a paid API call per ticket, so cap the train set at 2500
# rather than labeling the whole pool.
train, _ = train_test_split(
    train_pool, train_size=2500, stratify=train_pool["category"], random_state=42)

print("TRAIN\n", train["category"].value_counts())
print("\nTEST\n", test["category"].value_counts())

In [ ]:
import random
random.seed(42)

en = [
    "For reference, my account ends in {id}.",
    "Same account I always use, the one ending {id}.",
    "My account number is {id}.",
    "Please pull up account {id}.",
]
de = [
    "Zur Info, mein Konto endet auf {id}.",
    "Dasselbe Konto wie immer, endet auf {id}.",
    "Meine Kontonummer ist {id}.",
]

# The raw emails almost never state an account number, so inject one into ~70%.
# That gives account_id a real target to learn, and the other 30% teach it to return null.
def augment(frame, p_has_id=0.7):
    frame = frame.copy().reset_index(drop=True)
    bodies, ids = [], []
    for _, r in frame.iterrows():
        body = str(r["body"]).strip()
        if random.random() <= p_has_id:
            acct = str(random.randint(1000, 99999))
            tmpls = de if str(r["language"]).lower().startswith("de") else en
            body = body + " " + random.choice(tmpls).format(id=acct)
        else:
            acct = None
        bodies.append(body); ids.append(acct)
    frame["body_aug"] = bodies
    frame["account_id_true"] = ids
    return frame

train = augment(train)
test  = augment(test)

print("train w/ id:", train["account_id_true"].notna().sum(), "/", len(train))
print("test  w/ id:", test["account_id_true"].notna().sum(), "/", len(test))
print()
for _, r in train.sample(3, random_state=1).iterrows():
    print("true id:", r["account_id_true"], "| lang:", r["language"])
    print("…" + r["body_aug"][-140:])
    print("---")

In [ ]:
import os
from openai import OpenAI
client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")

r = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "Return ONLY JSON with fields: category, priority, account_id."},
        {"role": "user", "content": "My order ending 4471 never arrived, this is urgent."},
    ],
    temperature=0,
    response_format={"type": "json_object"},
)
print(r.choices[0].message.content)

In [ ]:
import json, time, os
from openai import OpenAI

client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")

CATEGORIES = ["Technical Support", "Product Support", "Customer Service",
              "Billing & Payments", "Returns & Exchanges"]
PRIORITIES = ["high", "medium", "low"]

SYSTEM = """You label customer support emails. Output ONLY a JSON object with exactly these fields:
- "category": exactly one of: Technical Support | Product Support | Customer Service | Billing & Payments | Returns & Exchanges
- "priority": exactly one of: high | medium | low
    high   = outage, blocked from using the product, security/payment failure, or urgent business impact
    medium = a real problem needing action, but not blocking or time-critical
    low    = general question, minor request, feedback, or no urgency
- "account_id": the customer's account number if one is stated in the email, as a string; otherwise null. Never invent one.
Output only the JSON object."""

def label_one(subject, body):
    user = f"Subject: {subject}\n\nEmail: {body}"
    for attempt in range(5):
        try:
            r = client.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": user}],
                temperature=0,
                response_format={"type": "json_object"},
            )
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            wait = 2 ** attempt
            print(f"  retry {attempt+1}/5 in {wait}s ({e})")
            time.sleep(wait)
    return None

OUT = "train_labeled.jsonl"
# The run is slow and costs money, so skip rows already in the file and append
# as we go. Flushing each line means a crash loses nothing.
done = set()
if os.path.exists(OUT):
    with open(OUT) as f:
        done = {json.loads(l)["idx"] for l in f}
print(f"resuming — already labeled: {len(done)}")

with open(OUT, "a") as f:
    for n, (idx, row) in enumerate(train.iterrows()):
        if idx in done:
            continue
        label = label_one(row["subject"], row["body_aug"])
        f.write(json.dumps({"idx": int(idx), "label": label}) + "\n")
        f.flush()
        if n % 50 == 0:
            print(f"{n}/{len(train)}")
print("done")

In [ ]:
import json, pandas as pd

labels = {}
with open("train_labeled.jsonl") as f:
    for l in f:
        rec = json.loads(l); labels[rec["idx"]] = rec["label"] or {}

t = train.copy()
t["pred_category"] = t.index.map(lambda i: labels.get(i, {}).get("category"))
t["pred_priority"] = t.index.map(lambda i: labels.get(i, {}).get("priority"))

print("off-vocab category:", (~t["pred_category"].isin(CATEGORIES)).sum())
print("off-vocab priority:", (~t["pred_priority"].isin(PRIORITIES)).sum())
print(f"\nteacher vs gold — category: {(t['pred_category']==t['category']).mean():.1%}"
      f" | priority: {(t['pred_priority']==t['priority']).mean():.1%}")
print("\npriority confusion (rows=gold, cols=teacher):")
print(pd.crosstab(t["priority"], t["pred_priority"]))

In [ ]:
mm = t[t["pred_category"] != t["category"]].sample(8, random_state=0)
for _, r in mm.iterrows():
    print(f"GOLD: {r['category']:<20} TEACHER: {r['pred_category']}")
    print(f"priority gold/teacher: {r['priority']} / {r['pred_priority']}")
    print(r["body_aug"][:280]); print("-"*60)

In [ ]:
assert "label_one" in dir() and "SYSTEM" in dir(), \
    "Re-run the train-labeling cell (defines SYSTEM + label_one) first, then run this."

import json, os

OUT = "test_labeled.jsonl"
done = set()
if os.path.exists(OUT):
    with open(OUT) as f:
        for l in f:
            rec = json.loads(l)
            if rec["label"] is not None:
                done.add(rec["idx"])
print(f"resuming — test rows done: {len(done)}")

with open(OUT, "a") as f:
    for n, (idx, row) in enumerate(test.iterrows()):
        if idx in done:
            continue
        label = label_one(row["subject"], row["body_aug"])
        f.write(json.dumps({"idx": int(idx), "label": label}) + "\n")
        f.flush()
        if n % 50 == 0:
            print(f"{n}/{len(test)}")
print("done — test set labeled")

In [ ]:
CATS = ["Technical Support","Product Support","Customer Service","Billing & Payments","Returns & Exchanges"]
PRIS = ["high","medium","low"]

tl = {}
with open("test_labeled.jsonl") as f:
    for l in f:
        rec = json.loads(l)
        if rec["label"]:
            tl[rec["idx"]] = rec["label"]

te = test.copy()
te["t_cat"] = te.index.map(lambda i: tl.get(i, {}).get("category"))
te["t_pri"] = te.index.map(lambda i: tl.get(i, {}).get("priority"))

print("labeled:", te["t_cat"].notna().sum(), "/", len(te))
print("off-vocab category:", (~te["t_cat"].isin(CATS)).sum())
print("off-vocab priority:", (~te["t_pri"].isin(PRIS)).sum())
print("failed (null) rows:", te["t_cat"].isna().sum())

In [ ]:
import json

SYSTEM_SHORT = (
    "Classify the support email. Respond with ONLY a JSON object with keys "
    '"category", "priority", and "account_id".\n'
    "category: one of Technical Support, Product Support, Customer Service, "
    "Billing & Payments, Returns & Exchanges.\n"
    "priority: one of high, medium, low.\n"
    'account_id: the account number stated in the email as a string, or null if none.'
)

labels = {}
with open("train_labeled.jsonl") as f:
    for l in f:
        rec = json.loads(l)
        if rec["label"]:
            labels[rec["idx"]] = rec["label"]

n_written, n_skipped = 0, 0
with open("train_chat.jsonl", "w") as out:
    for idx, row in train.iterrows():
        lab = labels.get(idx)
        # Drop rows where the teacher failed or returned a partial label. Fewer clean
        # examples beat training the model on nulls.
        if not lab or lab.get("category") is None or lab.get("priority") is None:
            n_skipped += 1
            continue
        target = {"category": lab["category"],
                  "priority": lab["priority"],
                  "account_id": lab.get("account_id")}
        messages = [
            {"role": "system", "content": SYSTEM_SHORT},
            {"role": "user", "content": f"Subject: {row['subject']}\n\nEmail: {row['body_aug']}"},
            {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)},
        ]
        out.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        n_written += 1

print(f"written: {n_written}  skipped(no label): {n_skipped}")
print(json.dumps(json.loads(open('train_chat.jsonl').readline()),
                 ensure_ascii=False, indent=2)[:900])

In [ ]:
import json, os

DATA_DIR = "/kaggle/input/datasets/abdulsamadzeeshan/tickets"

assert "test" in dir() and "body_aug" in test.columns, \
    "Run cell 0 (split) and cell 1 (augment) first so `test` has body_aug."

SYSTEM_SHORT = (
    "Classify the support email. Respond with ONLY a JSON object with keys "
    '"category", "priority", and "account_id".\n'
    "category: one of Technical Support, Product Support, Customer Service, "
    "Billing & Payments, Returns & Exchanges.\n"
    "priority: one of high, medium, low.\n"
    'account_id: the account number stated in the email as a string, or null if none.'
)

TL = "test_labeled.jsonl" if os.path.exists("test_labeled.jsonl") else f"{DATA_DIR}/test_labeled.jsonl"
labels = {}
with open(TL) as f:
    for l in f:
        rec = json.loads(l)
        if rec["label"]:
            labels[rec["idx"]] = rec["label"]
print("gold labels from:", TL, "| n =", len(labels))

n_written, n_skipped = 0, 0
with open("test_chat.jsonl", "w") as out:
    for idx, row in test.iterrows():
        lab = labels.get(idx)
        if not lab or lab.get("category") is None or lab.get("priority") is None:
            n_skipped += 1
            continue
        target = {"category": lab["category"],
                  "priority": lab["priority"],
                  "account_id": lab.get("account_id")}
        messages = [
            {"role": "system", "content": SYSTEM_SHORT},
            {"role": "user", "content": f"Subject: {row['subject']}\n\nEmail: {row['body_aug']}"},
            {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)},
        ]
        out.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        n_written += 1

print(f"written: {n_written}  skipped(no label): {n_skipped}  ->  test_chat.jsonl")
print(json.dumps(json.loads(open('test_chat.jsonl').readline()), ensure_ascii=False, indent=2)[:700])

In [ ]:
import torch, glob, os
print("GPU:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "*** NO GPU — enable it ***")
p = "/kaggle/working/test_chat.jsonl"
print("test_chat.jsonl:", os.path.exists(p), "| rows:", sum(1 for _ in open(p)) if os.path.exists(p) else 0)
print("\nloadable model folders in /kaggle/working:")
for d in sorted(glob.glob("/kaggle/working/*/")):
    cfg = os.path.exists(os.path.join(d, "config.json"))
    st  = bool(glob.glob(os.path.join(d, "*.safetensors")))
    if cfg and st:
        print(f"  ✓ {d}   (config.json + safetensors)")